# Exploratory Data Analysis for Q-Pilot System

This notebook explores the vehicle trajectory data used in the Q-Pilot system.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Import Q-Pilot modules
from src.dataset import generate_synthetic_trajectory
from src.preprocessing import DataPreprocessor
from src.feature_engineering import FeatureEngineer

%matplotlib inline
sns.set_style("whitegrid")

## 1. Data Generation and Loading

In [ ]:
# Generate synthetic trajectory data
raw_data = generate_synthetic_trajectory(1000)
print(f"Generated data shape: {raw_data.shape}")

# Create DataFrame
columns = ['x_position', 'y_position', 'velocity', 'acceleration', 'steering_angle', 'lane_id']
df = pd.DataFrame(raw_data, columns=columns)
df.head()

## 2. Data Exploration

In [ ]:
# Basic statistics
df.describe()

In [ ]:
# Check for missing values
df.isnull().sum()

In [ ]:
# Visualize distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for i, column in enumerate(columns):
    axes[i].hist(df[column], bins=30, alpha=0.7)
    axes[i].set_title(f'Distribution of {column}')
    axes[i].set_xlabel(column)
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 3. Trajectory Visualization

In [ ]:
# Plot sample trajectories
plt.figure(figsize=(12, 8))

# Plot multiple segments
for i in range(0, min(200, len(df)), 50):
    segment = df.iloc[i:min(i+50, len(df))]
    plt.plot(segment['x_position'], segment['y_position'], 
             marker='o', markersize=3, alpha=0.7, 
             label=f'Trajectory {i//50 + 1}')

plt.xlabel('X Position')
plt.ylabel('Y Position')
plt.title('Sample Vehicle Trajectories')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Feature Correlation Analysis

In [ ]:
# Correlation matrix
corr_matrix = df.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.show()

## 5. Sequence Analysis

In [ ]:
# Create sequences for analysis
seq_length = 5
pred_length = 3

X_data = []
y_data = []

for i in range(len(df) - seq_length - pred_length + 1):
    X_data.append(df.iloc[i:(i + seq_length)].values)
    y_data.append(df.iloc[(i + seq_length):(i + seq_length + pred_length)].values)

X_data = np.array(X_data)
y_data = np.array(y_data)

print(f"Input sequences shape: {X_data.shape}")
print(f"Target sequences shape: {y_data.shape}")

## 6. Feature Engineering Exploration

In [ ]:
# Apply feature engineering
engineer = FeatureEngineer()
enhanced_data = engineer.engineer_features(df.values)

print(f"Original features: {df.shape[1]}")
print(f"Enhanced features: {enhanced_data.shape[1]}")

# Show feature names
feature_names = engineer.get_feature_names()
extended_names = feature_names.copy()
extended_names.extend([
    'distance', 'cumulative_distance', 'direction',
    'jerk', 'speed_change_rate',
    'lane_change', 'lane_change_count'
])

print(f"\nFeature names: {extended_names}")

## 7. Data Preprocessing Pipeline

In [ ]:
# Apply preprocessing pipeline
from src.preprocessing import prepare_data_pipeline

(X_processed, y_processed), preprocessor = prepare_data_pipeline(
    df.values, seq_length=5, pred_length=3
)

print(f"Processed input shape: {X_processed.shape}")
print(f"Processed target shape: {y_processed.shape}")

# Show sample of processed data
print("\nSample processed input sequence:")
print(X_processed[0])

print("\nSample processed target sequence:")
print(y_processed[0])

## 8. Conclusion

This exploratory analysis shows that:

1. The synthetic trajectory data captures realistic vehicle movement patterns
2. Features have meaningful correlations that can be exploited by ML models
3. The sequence structure is suitable for time-series prediction tasks
4. Feature engineering can significantly expand the feature space
5. The preprocessing pipeline prepares data correctly for model training